# 实验七：网络损伤对 VR/360 视频的影响（Network Impairments on VR/360 Video）
## FMI Course · Kaggle Hands-on Lab

**课程**：未来媒体互联网（Future Media & Internet）&nbsp;|&nbsp; **预计时长**：~10 分钟 &nbsp;|&nbsp; **运行环境**：Kaggle Notebook（CPU） &nbsp;|&nbsp; **无需 GPU**

---

**与实验六的关系**：实验六建立了网络损伤（丢包/抖动/带宽骤降）对普通高清视频的基本影响规律。本实验使用与实验六**完全相同**的三种损伤模拟方法，但将其施加于 VR 等距矩形（Equirectangular）视频帧，展示相同的损伤强度在 VR 场景下为何会造成更严重的感知影响。

## 实验概述

VR/360 视频是沉浸式媒体的核心载体：用户佩戴头显后，可以自由转动视角观看全景画面，获得身临其境的体验。然而，VR 视频对网络损伤的敏感度远高于普通高清视频——同样是 5% 的丢包率，在 HD 视频中可能几乎不可察觉，在 VR 场景下却可能严重破坏用户的沉浸感甚至引发不适。

本实验复用实验六中定义的丢包、抖动、带宽骤降三种损伤模拟方法，将其施加于一个合成的等距矩形 360° 全景帧，并通过全景视图、视场角（FOV）裁切、中心放大三个层次的对比，直观展示 VR 格式为何会放大相同网络损伤的感知影响。

本实验对应课程中「沉浸式媒体与 VR/360 网络传输」部分的核心内容，是整个实验系列的收尾。

## 学习目标

完成本实验后，你应该能够：

1. 解释等距矩形投影（Equirectangular Projection）的基本原理，以及它为何会在两极区域放大压缩伪影；
2. 说明相比 HD 视频，VR/360 视频对带宽、帧率、时延提出了哪些更高的要求；
3. 理解视场角（Field of View, FOV）的概念，并解释为什么局部损伤在用户实际视场角内的影响会被放大；
4. 说明帧丢失如何导致运动眩晕（Motion Sickness），以及这与传统视频卡顿的本质区别；
5. 从实验结果中量化对比「相同损伤强度在 HD 与 VR 场景下」的感知差异，并联系到真实系统的缓解方案。

## 背景与基本原理

### 等距矩形投影（Equirectangular Projection, ERP）

VR/360 视频需要把一个完整的球形视野（用户可以看向任意方向）映射到一张普通的矩形图像上进行存储和传输，最常用的映射方式就是**等距矩形投影**——类似于世界地图的墨卡托投影原理，纬度方向（对应球面的上下方向）被线性拉伸映射到矩形的高度方向。这种映射方式会导致**极点区域（画面顶部和底部，对应球面的正上方和正下方）被严重拉伸**：球面上很小的一块区域，在等距矩形图像中会占据非常大的像素面积。这意味着压缩编码在拉伸后的极点区域会消耗大量码率，同时也让该区域的压缩伪影在映射后被相应放大。

### 为什么 VR 对网络的要求远高于 HD 视频

| 维度 | 普通 HD 视频 | VR/360 视频 |
|------|-------------|------------|
| 分辨率 | 1080p 左右 | 4K–8K（因为要覆盖整个球面，实际观看的那一小块视场角才能达到可接受清晰度） |
| 帧率 | 30 fps | 72–120 fps（避免头部转动时的眩晕感） |
| 时延要求 | 2–5 秒缓冲可接受 | < 20 毫秒（头动到画面更新的延迟，即 Motion-to-Photon Latency） |

### 视场角（Field of View, FOV）

用户佩戴 VR 头显时，每一时刻只能看到全景画面中的一小部分（通常水平约 100°、垂直约 90°），但传统的 VR 视频传输方案仍然需要把整个 360° 全景画面的数据都传输过去（即使大部分内容用户当下根本看不到）。这意味着：**任何发生在用户当前视场角内的损伤，都会被 100% 感知到**，而发生在视场角之外的损伤则完全不会被察觉——这与普通 HD 视频「整个画面都是用户注意力所在」的情况有本质不同。

### 帧丢失与运动眩晕（Motion Sickness）

人体通过视觉系统和前庭系统（负责平衡感）共同感知自身运动状态。在 VR 体验中，如果画面帧丢失或延迟导致视觉反馈与用户头部实际运动不同步，就会造成**视觉-前庭系统冲突**，这是运动眩晕的主要生理机制之一。普通视频的帧丢失只会造成短暂卡顿，用户体验下降但不会有生理不适；VR 视频的帧丢失却可能直接引发恶心、头晕等症状，是完全不同量级的问题。

### 相同损伤，不同感知：一个直观类比

普通 HD 视频中 5% 的丢包，损伤像素分散在整个 1920×1080 画面里，用户很可能根本注意不到；但在 VR 场景中，用户实际只关注约 100°×90° 的视场角窗口（大约相当于全景画面的 1/8），如果丢包恰好命中这个窗口内的区域，同样是 5% 的全局丢包率，在用户视场角内的「有效丢包密度」会显著更高，感知影响被明显放大。

## 实验设计

**与实验六的关系**：本实验直接复用实验六中定义的三个损伤模拟函数（`packet_loss` / `jitter` / `bw_drop`），代码实现完全相同，仅调整了函数内部的分块大小等参数以适配 VR 帧尺寸。这种设计使得两个实验之间的对比是「控制变量」的——损伤模拟方法不变，唯一变化的是画面内容从 HD 平面视频换成了 VR 等距矩形全景帧。

**VR 测试帧**：合成的等距矩形 360° 全景帧（960×480），包含水平线（地平线）、天空渐变（上半部分）、地面（下半部分）、三个不同位置的彩色物体（正前方红色、左侧绿色、右侧蓝色，用于标记不同视角方向）、以及网格参考线。

**三层对比视角**：① 完整 VR 全景帧；② 视场角裁切（模拟头显中用户实际看到的部分，270×480）；③ 中心区域放大（进一步聚焦视场角中央，观察细节损伤）。

**扩展验证**：额外模拟帧丢失导致的运动眩晕风险场景，以及在更接近真实内容的 VR 全景场景（含天空、远山、地面、建筑群）上重新验证结论。

## 运行环境说明

| 项目 | 说明 |
|------|------|
| 运行环境 | Kaggle Notebook |
| 计算资源 | CPU（无需 GPU） |
| 网络访问 | 不需要（Internet Off） |
| 主要依赖 | NumPy、Matplotlib、SciPy（Kaggle 已预装） |

直接点击「Run All」即可运行全部实验，无需上传数据或安装额外包。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
print('环境就绪 ✅')


## 步骤一：生成 VR 等距矩形测试帧

`create_vr_frame()` 构建了一张 960×480 的合成等距矩形全景帧，模拟真实 VR 内容（真实 VR 视频通常是 3840×1920 甚至更高分辨率，这里缩小尺寸以加快渲染速度）：

- **中心红色区域**：代表用户正前方视角（默认注视热点），是最需要保证质量的方向；
- **左侧绿色区域 / 右侧蓝色区域**：代表环视两侧的内容，用于观察不同水平方向的损伤影响；
- **水平线与网格**：作为空间参考线，帮助观察抖动导致的错位方向和幅度；
- **上下半部分的渐变**：分别代表天空和地面区域。

**观察重点**：留意画面顶部和底部区域——这正是等距矩形投影中被拉伸最严重、压缩伪影最容易被放大的区域。

In [ ]:
# Create a synthetic equirectangular VR frame
# VR 360 video is typically 4K+ (3840x1920 or higher)
# Scaled to 960x480 for fast rendering
EH, EW = 480, 960  # equirectangular dimensions

def create_vr_frame():
    """Create a synthetic equirectangular 360 frame with content regions"""
    frame = np.zeros((EH, EW, 3), dtype=np.float32)
    # Horizon line (equator)
    frame[EH//2-2:EH//2+2, :] = 0.8
    # Sky gradient (top half)
    for y in range(EH//2):
        frame[y, :] = (0.3 + 0.5*y/(EH//2), 0.4 + 0.4*y/(EH//2), 0.6 + 0.3*y/(EH//2))
    # Ground (bottom half)
    frame[EH//2:, :] = (0.1, 0.3, 0.1)
    # Objects at different positions (simulating 360 content)
    # Center object (where viewer is likely looking)
    cx, cy = EW//2, EH//2
    for y in range(cy-60, cy+60):
        for x in range(cx-80, cx+80):
            if (x-cx)**2 + (y-cy)**2 < 60**2:
                frame[y, x] = (0.9, 0.2, 0.2)
    # Left object
    for y in range(cy-30, cy+30):
        for x in range(cx-300, cx-200):
            if (x-(cx-250))**2 + (y-cy)**2 < 30**2:
                frame[y, x] = (0.2, 0.9, 0.2)
    # Right object
    for y in range(cy-30, cy+30):
        for x in range(cx+200, cx+300):
            if (x-(cx+250))**2 + (y-cy)**2 < 30**2:
                frame[y, x] = (0.2, 0.2, 0.9)
    # Grid lines for spatial reference
    for x in range(0, EW, 120):
        frame[:, x] *= 0.7
    for y in range(0, EH, 60):
        frame[y, :] *= 0.7
    return frame

vr_frame = create_vr_frame()
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(np.clip(vr_frame, 0, 1))
ax.set_title('VR Equirectangular Frame (960x480)\nRed=Front | Green=Left | Blue=Right', fontsize=13)
ax.axis('off')
plt.show()
print("等距矩形投影会拉伸画面两极（顶部/底部），导致极点区域的压缩伪影被放大。")
print("正前方（红色区域）是用户注视热点，对损伤最敏感。")


## 步骤二：网络损伤模拟函数（与实验六完全相同的实现）

这部分代码与实验六中的损伤模拟函数**逻辑完全一致**（仅分块参数按 VR 帧尺寸做了适配）：`packet_loss()` 按块随机丢弃填充灰色，`jitter()` 逐行随机水平位移，`bw_drop()` 先降采样再放大模拟带宽不足。

由于两个实验使用相同的损伤模拟方法，我们可以放心地把接下来观察到的「感知影响差异」完全归因于**画面内容从 HD 平面视频变成了 VR 全景视频**，而不是损伤模拟方式本身的不同。

In [ ]:
# Network impairment simulators (frame-size-aware)
def packet_loss(frame, rate):
    result = frame.copy()
    H, W = frame.shape[:2]
    bh, bw = 24, 40
    nb_h, nb_w = H//bh, W//bw
    mask = np.random.random((nb_h, nb_w)) < rate
    for i in range(nb_h):
        for j in range(nb_w):
            if mask[i,j]:
                i1, i2 = i*bh, min((i+1)*bh, H)
                j1, j2 = j*bw, min((j+1)*bw, W)
                result[i1:i2, j1:j2] = 0.5
    return result

def jitter(frame, px):
    result = frame.copy()
    H = frame.shape[0]
    shifts = (np.random.randn(H)*px).astype(int)
    for y in range(H):
        result[y] = np.roll(result[y], shifts[y], axis=0)
    return result

def bw_drop(frame, scale):
    H, W = frame.shape[:2]
    h2, w2 = int(H*scale), int(W*scale)
    low = ndimage.zoom(frame, (scale,scale,1), order=1)
    up = ndimage.zoom(low, (1/scale,1/scale,1), order=1)
    return np.clip(up[:H,:W], 0, 1)

print("损伤模拟器就绪（已适配 VR 帧尺寸）")


## 步骤三：三层视角对比——全景帧 / 视场角裁切 / 中心放大

下方代码构建一个 3×3 的对比网格：每一列对应一种情况（原始画面 / 5% 丢包 / 5px 抖动），每一行对应一个观察层次：

- **第一行**：完整的 VR 全景帧；
- **第二行**：从全景帧中裁切出的视场角区域（270×480），代表用户头显中实际看到的画面范围；
- **第三行**：在视场角基础上进一步放大中心区域，聚焦观察细节损伤。

**观察重点**：对比同一种损伤（如 5% 丢包）在第一行（全景视角，损伤看起来「稀疏」）与第二、三行（视场角/中心放大，损伤变得「集中且显著」）之间的观感差异——这正是「局部化的视场角放大了损伤感知影响」这一核心结论的直接体现。

In [ ]:
# 对比：相同网络损伤对 VR 和 HD 的不同影响
np.random.seed(42)

# 从 VR 全景帧中裁切视场角区域（模拟头显中实际看到的画面）
fov_h, fov_w = 270, 480
cy, cx = EH//2, EW//2
hd_crop = vr_frame[cy-fov_h//2:cy+fov_h//2, cx-fov_w//2:cx+fov_w//2]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

impairments = [
    ('Original', lambda f: f),
    ('Loss 5%', lambda f: packet_loss(f, 0.05)),
    ('Jitter 5px', lambda f: jitter(f, 5)),
]

for j, (label, func) in enumerate(impairments):
    # 完整 VR 帧
    axes[0, j].imshow(np.clip(func(vr_frame), 0, 1))
    axes[0, j].set_title('Full VR Frame: {}'.format(label), fontsize=11)
    axes[0, j].axis('off')
    # 视场角裁切（用户实际看到的部分）
    crop = np.clip(func(hd_crop), 0, 1)
    axes[1, j].imshow(crop)
    axes[1, j].set_title('FOV Crop: {}'.format(label), fontsize=11)
    axes[1, j].axis('off')
    # 中心物体放大
    ch, cw = crop.shape[0]//2, crop.shape[1]//2
    r = 60
    z = crop[max(0,ch-r):min(crop.shape[0],ch+r), max(0,cw-r):min(crop.shape[1],cw+r)]
    axes[2, j].imshow(z)
    axes[2, j].set_title('Center Zoom: {}'.format(label), fontsize=11)
    axes[2, j].axis('off')

plt.suptitle('VR vs HD: Same Impairment, Different Perceptual Impact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("观察要点：")
print("1. VR 全景帧中，损伤在画面各处可见")
print("2. 在视场角裁切（用户实际观看区域）中，损伤更加集中和明显")
print("3. 中心物体放大后，即使 5% 的丢包也会破坏关键视觉信息")
print("4. VR 需要比 HD 高 4-8 倍的带宽才能达到同等感知质量")


## 步骤四：帧丢失与运动眩晕风险模拟

这部分代码进一步展示了 VR 场景独有的时延敏感性问题：`frame_drop_sim()` 模拟帧丢失/延迟导致的画面质量下降，代码依次展示「流畅 60fps」「50% 帧丢失（有运动眩晕风险）」「严重损伤：带宽骤降 50% + 丢包 10%（VR 体验完全崩溃）」三种情况。

**观察重点**：结合背景原理中提到的「视觉-前庭系统冲突」机制，理解为什么 VR 场景中的帧丢失不仅仅是「画面卡了一下」这么简单，而是可能直接引发生理层面的不适。

In [ ]:
# VR 时延敏感性对比
def frame_drop_sim(frame, drop_ratio):
    return frame * (1.0 - drop_ratio * 0.3)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(np.clip(vr_frame, 0, 1))
axes[0].set_title('Smooth VR (60 fps)', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(np.clip(frame_drop_sim(vr_frame, 0.5), 0, 1))
axes[1].set_title('Frame Drops 50%\n⚠️ Motion Sickness Risk!', fontsize=12)
axes[1].axis('off')

axes[2].imshow(np.clip(packet_loss(bw_drop(vr_frame, 0.5), 0.1), 0, 1))
axes[2].set_title('Severe: BW-50% + Loss 10%\n❌ VR Experience Broken', fontsize=12)
axes[2].axis('off')

plt.suptitle('VR Quality Degradation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n为什么 VR 对网络损伤更敏感：")
print("1. 更高分辨率（4K-8K vs 1080p）-> 每帧数据量更大")
print("2. 超低时延要求（<20ms vs 流媒体 2-5s 缓冲）")
print("3. 帧丢失直接导致运动眩晕（前庭-视觉冲突）")
print("4. 等距矩形投影在两极浪费大量码率")
print("5. 注视点渲染假设传输完美无缺")


## 步骤五：在真实感 VR 场景上验证结论

与实验六类似，为了确认前面基于人工合成全景帧得到的结论具有普适性，这一步构建了一个更接近真实 VR 内容的全景场景：`create_realistic_vr()` 生成包含天空渐变、起伏远山轮廓、地面纹理、以及随机散布的建筑群（模拟 360° 城市漫游场景）的画面，并标注了中心注视热点区域。

随后在这个真实感场景的视场角裁切上，对比「理想无损伤」「5% 丢包」「3px 抖动」「带宽骤降 50%+丢包 5%」四种情况。

**观察重点**：即使画面内容从人工几何图案换成更自然的城市全景，「视场角内损伤被放大感知」以及「多种损伤叠加时体验迅速崩溃」这两个核心结论依然成立。

In [ ]:
# 模拟更真实的 VR 全景场景
def create_realistic_vr():
    EH, EW = 480, 960
    frame = np.zeros((EH, EW, 3), dtype=np.float32)
    # 天空
    for y in range(EH//2):
        t = y/(EH//2)
        frame[y, :] = (0.2+0.5*t, 0.3+0.4*t, 0.5+0.4*t)
    # 远山（起伏轮廓）
    for x in range(EW):
        h = int(EH*0.35 + np.sin(x*0.008)*30 + np.sin(x*0.02)*20 + np.sin(x*0.05)*10)
        frame[h:EH//2+20, x] = (0.12, 0.28, 0.12)
    # 地面
    for y in range(EH//2+20, EH):
        v = 0.18 + 0.1*np.sin(y*0.04)
        frame[y, :] = (v, 0.22+v*0.4, v*0.4)
    # 建筑物散布在 360 空间中
    import random; random.seed(7)
    for _ in range(12):
        bx = random.randint(20, EW-100)
        bw = random.randint(40, 90)
        bh = random.randint(50, 130)
        by = int(EH*0.35 - bh + random.randint(-20, 30))
        c = random.uniform(0.3, 0.7)
        by = max(0, by)
        top = min(by, EH//2+20)
        bot = min(top+bh, EH//2+20)
        frame[top:bot, bx:bx+bw] = (c*0.6, c*0.5, c*0.4)
    # 中心区域标注（用户注视热点）
    cx, cy = EW//2, EH//2
    rr = 100
    for y in range(cy-rr, cy+rr):
        for x in range(cx-rr, cx+rr):
            if (x-cx)**2 + (y-cy)**2 < rr**2:
                if 0 <= y < EH and 0 <= x < EW:
                    frame[y, x] *= 1.2
    return np.clip(frame, 0, 1)

real_vr = create_realistic_vr()
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(real_vr)
ax.set_title('Realistic VR Equirectangular Scene')
ax.axis('off')
plt.show()
print("全景场景包含：天空、远山、地面、散布的建筑群、中心注视热点")


In [ ]:
# 在真实感 VR 场景上对比不同网络损伤
np.random.seed(99)
# 裁切视场角区域
cy, cx = EH//2, EW//2
fov = real_vr[cy-135:cy+135, cx-240:cx+240]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
scenarios = [
    ('Ideal VR (No Impairment)', fov),
    ('Loss 5%', packet_loss(fov, 0.05)),
    ('Jitter 3px', jitter(fov, 3)),
    ('BW-50% + Loss 5%', packet_loss(bw_drop(fov, 0.5), 0.05)),
]
for j, (label, img) in enumerate(scenarios):
    axes[j].imshow(np.clip(img, 0, 1))
    axes[j].set_title(label, fontsize=12, fontweight='bold')
    axes[j].axis('off')
plt.suptitle('VR Viewport: Network Impairment Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nVR 体验对网络的严苛要求总结：")
print("1. 5% 丢包在 HD 中可接受，在 VR 中已明显破坏体验")
print("2. 3px 抖动对 480px 宽视场角影响显著（占比 0.6%）")
print("3. 带宽下降 50% 叠加丢包时，VR 画面已不可用")
print("4. 这解释了为什么 VR 需要 5G/WiFi 6E 级别的网络支撑")


## 实验结果与分析

### 相同损伤、不同格式的感知差异

综合三层视角对比与真实感场景验证，可以得出一个量化的核心结论：**5% 的丢包率在普通 HD 视频的完整画面中几乎难以察觉，但在 VR 视场角内（约占全景画面 1/8 到 1/6 的区域）却清晰可见、明显影响观感**。这是因为用户在 VR 中只关注视场角内的内容，损伤一旦落入这个窗口，就无法像在 HD 全画幅中那样被「稀释」到视觉不敏感的区域。

### 极点拉伸对压缩伪影的放大效应

等距矩形投影对画面顶部和底部（对应球面南北极附近）的拉伸，意味着这些区域在编码时消耗了不成比例的码率，也意味着一旦发生丢包或带宽不足，这些被拉伸区域的伪影会在视觉上被相应放大。虽然用户较少直接注视正上方或正下方，但在头显自由转动、或应用场景涉及垂直方向内容（如仰望天空、俯瞰地面）时，这一效应会直接影响体验质量。

### 帧丢失严重程度的本质差异

普通视频的帧丢失表现为「卡顿-等待缓冲」的体验降级，用户虽然不满但不会有生理不适；VR 视频的帧丢失/延迟直接关联「视觉-前庭系统冲突」这一运动眩晕的生理机制，属于「零容忍」级别的问题——这解释了为什么 VR 系统对时延的要求（<20ms）比传统流媒体（2-5秒缓冲）严格了两个数量级以上。

## 从实验到实际系统

本实验揭示的核心问题——VR 对网络损伤的敏感度远高于 HD 视频——正是推动以下关键技术发展的原因：

- **注视点渲染（Foveated Rendering）**：利用眼动追踪技术，只在用户视线焦点区域渲染/传输最高质量的内容，视场角边缘区域则可以适当降低质量，在不明显影响体验的前提下大幅节省带宽；
- **视口依赖流式传输（Viewport-Dependent Streaming）**：类似于本实验中「只有视场角内的损伤才重要」这一发现的工程应用——系统只为用户当前视场角对应的区域传输高质量数据，非视场角区域传输低质量或不传输，是当前 360° 视频流媒体的主流优化方案；
- **5G 边缘计算**：通过将计算和存储资源下沉到网络边缘，大幅降低端到端时延，满足 VR 对超低时延（<20ms）的严苛要求；
- **Apple Vision Pro 等新一代设备**：采用了专用的低时延无线传输链路和本地渲染优化，试图在无线条件下也能满足 VR 对时延和带宽的双重严苛要求。

---

## 本实验小结

通过本实验，你应该掌握以下核心结论：

1. **等距矩形投影会拉伸并放大两极区域的压缩伪影**：这是 VR 视频编码面临的独特几何挑战；
2. **VR 对带宽、帧率、时延的要求全面高于传统 HD 视频**：分辨率需求高 4-8 倍，帧率需求高 2-4 倍，时延要求严格两个数量级；
3. **视场角机制会放大局部损伤的感知影响**：相同的全局损伤率，在用户实际视场角内的「有效影响密度」远高于 HD 全画幅；
4. **VR 中的帧丢失直接关联运动眩晕**：这是与传统视频卡顿完全不同性质的问题，容错空间极小；
5. **真实系统通过注视点渲染、视口依赖传输等技术专门应对这些挑战**：这些技术都是针对本实验揭示的核心矛盾而设计的。

---

## 思考与拓展

以下问题没有唯一答案，鼓励你修改代码并重新运行：

1. **改变视场角位置**：修改 FOV 裁切的中心坐标，模拟用户望向两极（画面顶部/底部）而非赤道方向，观察拉伸区域的伪影是否比望向赤道方向更严重。
2. **实现简易注视点渲染**：编写代码对中心「注视」区域保持原始质量，对边缘区域应用更强的 `bw_drop()` 模糊，模拟注视点渲染的基本效果。
3. **量化敏感度差距**：复用实验三的 PSNR/SSIM 实现，分别对实验六的 HD 帧和本实验的 VR 帧在相同损伤强度下计算指标，用数值验证「VR 更敏感」这一定性结论。
4. **组合极端损伤**：尝试叠加更多种类型的损伤（丢包+抖动+带宽骤降三者同时），观察 VR 场景下的体验是否比 HD 场景下崩溃得更快。
5. **对比不同视场角大小**：修改 `fov_h, fov_w` 参数，模拟更窄或更宽的头显视场角设置，观察视场角大小如何影响损伤的感知放大程度。

---

← [实验六：网络损伤对高清视频的影响](https://www.kaggle.com/code/guopingtan/fmi-demo-6-network-impairments-on-hd-video) &nbsp;|&nbsp; 🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University